In [14]:
from huggingface_hub import login
login()

In [6]:
import os, time, traceback
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"   # Windows noise

from pathlib import Path
from collections import defaultdict
from datasets import load_dataset
from huggingface_hub import snapshot_download
from rich.console import Console
from rich.panel import Panel
from rich.table import Table
from rich.rule import Rule

console = Console()

REPO = "nvidia/Nemotron-RL-Instruction-Following-Structured-Outputs-v2"
CACHE_DIR = r"C:\hf_cache"
SAVE_ROOT = Path(r"C:\Users\krish\OneDrive\Desktop\all-about-slms\structure-output-slm\dataset2")
SAVE_ROOT.mkdir(parents=True, exist_ok=True)

console.print(Panel.fit(
    f"[bold cyan]Dataset:[/] {REPO}\n"
    f"[bold cyan]Cache:[/]   {CACHE_DIR}\n"
    f"[bold cyan]Save to:[/] {SAVE_ROOT}",
    title="[bold magenta]🚀 HF Downloader[/]", border_style="magenta",
))

# 1. Download raw repo files (only parquet + metadata, skips the broken card parsing)
console.print(Rule("[bold yellow]Step 1: Downloading repo files[/]"))
t0 = time.time()
local_dir = Path(snapshot_download(
    REPO, repo_type="dataset", cache_dir=CACHE_DIR,
    allow_patterns=["*.parquet", "*.json", "*.jsonl", "README.md"],
))
console.print(f"[bold green]✔ Downloaded in {time.time() - t0:.1f}s[/] → [dim]{local_dir}[/]")

# 2. Group parquet files (by folder, or by filename if they sit in the root)
files = sorted(local_dir.rglob("*.parquet"))
if not files:
    console.print("[bold red]✘ No parquet files found. Check the repo layout.[/]")
    raise SystemExit(1)

groups = defaultdict(list)
for f in files:
    rel = f.relative_to(local_dir)
    key = str(rel.parent).replace("\\", "_").replace("/", "_") if str(rel.parent) != "." else rel.stem
    groups[key].append(f)

tree = Table(title="Discovered groups", header_style="bold cyan", border_style="blue")
tree.add_column("Group", style="bold green")
tree.add_column("# files", justify="right", style="yellow")
tree.add_column("Example file", style="dim")
for k, v in groups.items():
    tree.add_row(k, str(len(v)), v[0].name)
console.print(tree)

# 3. Load + save each group separately
summary = Table(title="Results", header_style="bold cyan", border_style="blue")
summary.add_column("Group", style="bold green")
summary.add_column("Status")
summary.add_column("Rows", justify="right", style="yellow")
summary.add_column("Columns", style="white")

for name, paths in groups.items():
    console.print(Rule(f"[bold magenta]{name}[/]"))
    try:
        t0 = time.time()
        console.print(f"[yellow]⬇  Loading {len(paths)} file(s)...[/]")
        ds = load_dataset("parquet", data_files=[str(p) for p in paths], cache_dir=CACHE_DIR)

        for split, d in ds.items():
            console.print(f"  [cyan]{split}[/]: [yellow]{d.num_rows:,}[/] rows | cols: [white]{d.column_names}[/]")

        out = SAVE_ROOT / name
        console.print(f"[yellow]💾 Saving → {out}[/]")
        ds.save_to_disk(str(out))

        size_mb = sum(f.stat().st_size for f in out.rglob("*") if f.is_file()) / 1024**2
        console.print(f"[bold green]✔ Done in {time.time() - t0:.1f}s[/] [dim]({size_mb:,.1f} MB)[/]")

        first = next(iter(ds.values()))
        summary.add_row(name, "[green]✔ OK[/]", f"{first.num_rows:,}", ", ".join(first.column_names))
    except Exception as e:
        console.print(f"[bold red]✘ Failed:[/] {e}")
        console.print(traceback.format_exc(), style="dim red")
        summary.add_row(name, "[red]✘ FAILED[/]", "-", str(e)[:60])

console.print(Rule("[bold green]Finished[/]"))
console.print(summary)

╭────────────────────────────────── 🚀 HF Downloader ───────────────────────────────────╮
│ Dataset: nvidia/Nemotron-RL-Instruction-Following-Structured-Outputs-v2               │
│ Cache:   C:\hf_cache                                                                  │
│ Save to: C:\Users\krish\OneDrive\Desktop\all-about-slms\structure-output-slm\dataset2 │
╰───────────────────────────────────────────────────────────────────────────────────────╯

───────────────────────────────────────── Step 1: Downloading repo files ──────────────────────────────────────────

Fetching 4 files: 100%|██████████| 4/4 [01:15<00:00, 18.85s/it]


✔ Downloaded in 75.9s → 
C:\hf_cache\datasets--nvidia--Nemotron-RL-Instruction-Following-Structured-Outputs-v2\snapshots\b0187e1802944a6dd01
560f74d124ce9c19fe0e8

                         Discovered groups                          
┏━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Group                   ┃ # files ┃ Example file                 ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ direct_generation       │       1 │ train-00000-of-00001.parquet │
│ diversified_tasks       │       1 │ train-00000-of-00001.parquet │
│ tool_calling_extraction │       1 │ train-00000-of-00001.parquet │
└─────────────────────────┴─────────┴──────────────────────────────┘

──────────────────────────────────────────────── direct_generation ────────────────────────────────────────────────

... (18 outputs left out)

Saving the dataset (1/1 shards): 100%|██████████| 8783/8783 [00:00<00:00, 21368.68 examples/s]


✔ Done in 1.2s (228.4 MB)

──────────────────────────────────────────────────── Finished ─────────────────────────────────────────────────────

                                                      Results                                                      
┏━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Group                   ┃ Status ┃   Rows ┃ Columns                                                             ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ direct_generation       │ ✔ OK   │ 28,145 │ responses_create_params, schema_str, schema_type,                   │
│                         │        │        │ schema_fields_count, agent_ref                                      │
│ diversified_tasks       │ ✔ OK   │ 25,768 │ responses_create_params, schema_str, schema_type, problem_type,     │
│                         │        │        │ schema_repr, source_format, num_turns, source_record_id, agent_ref  │
│ tool_calling_extraction │ ✔ OK   │  8,783 │ responses_create_params, schema_str, schema_type, response_mode,    │
│                         │        │        │ problem_type, schema_repr, source_format, source_schema_type,       │
│                         │        │        │ tool_choice, parallel_tool_calls, tool_name, tool_schema_mode,      │
│                         │        │        │ tool_payload_key, tool_name_style, distractor_style,                │
│                         │        │        │ tool_union_mode, num_turns, num_tools, num_distractors,             │
│                         │        │        │ has_distractors, instruction_layout, instruction_detail_level,      │
│                         │        │        │ system_instruction_style, source_record_id, agent_ref               │
└─────────────────────────┴────────┴────────┴─────────────────────────────────────────────────────────────────────┘

In [8]:
import os, time, json, shutil
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"   # Windows noise

from pathlib import Path
from collections import Counter
from datasets import Dataset, Features, Value
from huggingface_hub import snapshot_download
from rich.console import Console
from rich.panel import Panel
from rich.table import Table
from rich.rule import Rule
from rich.progress import Progress, SpinnerColumn, TextColumn, TimeElapsedColumn

console = Console()

REPO = "Emulated-Inc/structured-output-training-pool"
CACHE_DIR = r"C:\hf_cache"
SAVE_ROOT = Path(r"C:\Users\krish\OneDrive\Desktop\all-about-slms\structure-output-slm\dataset3")
SAVE_ROOT.mkdir(parents=True, exist_ok=True)

console.print(Panel.fit(
    f"[bold cyan]Dataset:[/] {REPO}\n"
    f"[bold cyan]Cache:[/]   {CACHE_DIR}\n"
    f"[bold cyan]Save to:[/] {SAVE_ROOT}",
    title="[bold magenta]🚀 HF Downloader[/]", border_style="magenta",
))

# 1. Download only pool.jsonl + README
console.print(Rule("[bold yellow]Step 1: Downloading repo files[/]"))
t0 = time.time()
local_dir = Path(snapshot_download(
    REPO, repo_type="dataset", cache_dir=CACHE_DIR,
    allow_patterns=["*pool.jsonl", "README.md"],
))
console.print(f"[bold green]✔ Downloaded in {time.time() - t0:.1f}s[/] → [dim]{local_dir}[/]")

# 2. Find pool.jsonl
console.print(Rule("[bold yellow]Step 2: Locating pool.jsonl[/]"))
files = sorted(local_dir.rglob("pool.jsonl"))
if not files:
    console.print("[bold red]✘ No pool.jsonl found. Check the repo layout.[/]")
    raise SystemExit(1)
src = files[0]
console.print(f"[bold green]✔ Found:[/] [dim]{src}[/] [yellow]({src.stat().st_size / 1024**2:,.1f} MB)[/]")

# 3. Copy the file + stream through it (no Arrow, so no type inference)
console.print(Rule("[bold magenta]Step 3: Copying + scanning pool.jsonl[/]"))
out = SAVE_ROOT / "pool.jsonl"

t0 = time.time()
console.print(f"[yellow]📄 Copying →[/] {out}")
shutil.copy2(src, out)
console.print(f"[bold green]✔ Copied in {time.time() - t0:.1f}s[/] [dim]({out.stat().st_size / 1024**2:,.1f} MB)[/]")

by_shape, by_source, by_prov, by_licence = Counter(), Counter(), Counter(), Counter()
total, bad = 0, 0

with Progress(
    SpinnerColumn(style="magenta"),
    TextColumn("[cyan]Scanning rows:[/] [yellow]{task.completed:,}[/]"),
    TimeElapsedColumn(),
    console=console,
) as prog:
    task = prog.add_task("scan", total=None)
    with open(out, encoding="utf-8") as f:
        for line in f:
            try:
                r = json.loads(line)
            except json.JSONDecodeError:
                bad += 1
                continue
            total += 1
            by_shape[r.get("row_shape")] += 1
            by_source[r.get("source")] += 1
            by_prov[r.get("provenance_class")] += 1
            by_licence[r.get("licence")] += 1
            prog.advance(task)

def show(title, counter, color):
    t = Table(title=title, header_style=f"bold {color}", border_style=color)
    t.add_column("Value", style="bold green")
    t.add_column("Rows", justify="right", style="yellow")
    t.add_column("%", justify="right", style="dim")
    for k, v in counter.most_common():
        t.add_row(str(k), f"{v:,}", f"{v / total * 100:.1f}")
    console.print(t)

console.print(Rule("[bold green]Summary[/]"))
console.print(f"[bold cyan]Total rows:[/] [yellow]{total:,}[/]   [bold red]Bad lines:[/] {bad}")
show("row_shape", by_shape, "cyan")
show("source", by_source, "magenta")
show("provenance_class", by_prov, "blue")
show("licence", by_licence, "yellow")

# 4. Optional: build a proper HF Dataset (schema/document stored as JSON strings)
console.print(Rule("[bold magenta]Step 4: Building HF Dataset[/]"))

COLS = ["id", "request", "schema", "document", "row_shape", "schema_origin", "source",
        "source_repo", "source_revision", "source_subset", "source_split", "source_id",
        "provenance_class", "licence"]

def gen():
    with open(out, encoding="utf-8") as f:
        for line in f:
            r = json.loads(line)
            yield {c: (None if r.get(c) is None else r[c] if isinstance(r[c], str) else json.dumps(r[c])) for c in COLS}

t0 = time.time()
ds = Dataset.from_generator(gen, features=Features({c: Value("string") for c in COLS}), cache_dir=CACHE_DIR)
hf_out = SAVE_ROOT / "hf_dataset"
ds.save_to_disk(str(hf_out))
console.print(f"[bold green]✔ Saved HF dataset in {time.time() - t0:.1f}s:[/] [yellow]{ds.num_rows:,}[/] rows → [dim]{hf_out}[/]")

console.print(Panel.fit(
    f"[bold green]All done![/]\n"
    f"[cyan]Raw:[/] {out}\n"
    f"[cyan]HF: [/] {hf_out}",
    border_style="green",
))

╭────────────────────────────────── 🚀 HF Downloader ───────────────────────────────────╮
│ Dataset: Emulated-Inc/structured-output-training-pool                                 │
│ Cache:   C:\hf_cache                                                                  │
│ Save to: C:\Users\krish\OneDrive\Desktop\all-about-slms\structure-output-slm\dataset3 │
╰───────────────────────────────────────────────────────────────────────────────────────╯

───────────────────────────────────────── Step 1: Downloading repo files ──────────────────────────────────────────

Fetching 2 files: 100%|██████████| 2/2 [00:00<00:00, 567.03it/s]


✔ Downloaded in 0.3s → 
C:\hf_cache\datasets--Emulated-Inc--structured-output-training-pool\snapshots\c3dd6ab2b4da166f3bb29932be2f76cff5631
253

─────────────────────────────────────────── Step 2: Locating pool.jsonl ───────────────────────────────────────────

✔ Found: 
C:\hf_cache\datasets--Emulated-Inc--structured-output-training-pool\snapshots\c3dd6ab2b4da166f3bb29932be2f76cff5631
253\pool.jsonl (289.5 MB)

... (10 outputs left out)

─────────────────────────────────────────── Step 4: Building HF Dataset ───────────────────────────────────────────

Generating train split: 114103 examples [00:19, 5897.47 examples/s]
Saving the dataset (1/1 shards): 100%|██████████| 114103/114103 [00:00<00:00, 227151.13 examples/s]


✔ Saved HF dataset in 20.0s: 114,103 rows → 
C:\Users\krish\OneDrive\Desktop\all-about-slms\structure-output-slm\dataset3\hf_dataset

╭──────────────────────────────────────────────────────────────────────────────────────────────╮
│ All done!                                                                                    │
│ Raw: C:\Users\krish\OneDrive\Desktop\all-about-slms\structure-output-slm\dataset3\pool.jsonl │
│ HF:  C:\Users\krish\OneDrive\Desktop\all-about-slms\structure-output-slm\dataset3\hf_dataset │
╰──────────────────────────────────────────────────────────────────────────────────────────────╯

In [9]:
import os, time, shutil, traceback
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"   # Windows noise

from pathlib import Path
from datasets import load_dataset
from huggingface_hub import HfApi, snapshot_download
from rich.console import Console
from rich.panel import Panel
from rich.table import Table
from rich.rule import Rule
from rich.syntax import Syntax

console = Console()

REPO = "stindardlogic/structured-output-sft-100k"
CACHE_DIR = r"C:\hf_cache"
SAVE_ROOT = Path(r"C:\Users\krish\OneDrive\Desktop\all-about-slms\structure-output-slm\dataset4")
SAVE_ROOT.mkdir(parents=True, exist_ok=True)

console.print(Panel.fit(
    f"[bold cyan]Dataset:[/] {REPO}\n"
    f"[bold cyan]Cache:[/]   {CACHE_DIR}\n"
    f"[bold cyan]Save to:[/] {SAVE_ROOT}",
    title="[bold magenta]🚀 HF Downloader[/]", border_style="magenta",
))

# 1. Peek at the repo files
console.print(Rule("[bold yellow]Step 1: Repo contents[/]"))
info = HfApi().dataset_info(REPO, files_metadata=True)
ftable = Table(title=f"{len(info.siblings)} files", header_style="bold cyan", border_style="blue")
ftable.add_column("File", style="bold green")
ftable.add_column("Size (MB)", justify="right", style="yellow")
for s in sorted(info.siblings, key=lambda s: s.rfilename):
    ftable.add_row(s.rfilename, f"{(s.size or 0) / 1024**2:,.1f}")
console.print(ftable)

# 2. Try the normal load
console.print(Rule("[bold yellow]Step 2: load_dataset[/]"))
t0 = time.time()
try:
    ds = load_dataset(REPO, cache_dir=CACHE_DIR)
    console.print(f"[bold green]✔ Loaded in {time.time() - t0:.1f}s[/]")

    table = Table(title="Dataset Summary", header_style="bold cyan", border_style="blue")
    table.add_column("Split", style="bold green")
    table.add_column("Rows", justify="right", style="yellow")
    table.add_column("Columns", style="white")
    for split, d in ds.items():
        table.add_row(split, f"{d.num_rows:,}", ", ".join(d.column_names))
    console.print(table)

    # sample row
    first = next(iter(ds.values()))
    console.print(Rule("[bold magenta]Sample row[/]"))
    for k, v in first[0].items():
        console.print(f"[bold cyan]{k}[/]: [white]{str(v)[:300]}[/]")

    # 3. Save
    console.print(Rule("[bold yellow]Step 3: Saving[/]"))
    t0 = time.time()
    out = SAVE_ROOT / "hf_dataset"
    ds.save_to_disk(str(out))
    size_mb = sum(f.stat().st_size for f in out.rglob("*") if f.is_file()) / 1024**2
    console.print(f"[bold green]✔ Saved in {time.time() - t0:.1f}s[/] [dim]({size_mb:,.1f} MB)[/] → {out}")

except Exception as e:
    # Fallback: grab raw files so you still have the data
    console.print(f"[bold red]✘ load_dataset failed:[/] {str(e)[:300]}")
    console.print(traceback.format_exc(), style="dim red")
    console.print(Rule("[bold yellow]Fallback: downloading raw files[/]"))

    t0 = time.time()
    local_dir = Path(snapshot_download(REPO, repo_type="dataset", cache_dir=CACHE_DIR))
    raw_out = SAVE_ROOT / "raw"
    shutil.copytree(local_dir, raw_out, dirs_exist_ok=True)
    console.print(f"[bold green]✔ Raw files copied in {time.time() - t0:.1f}s[/] → {raw_out}")
    console.print("[yellow]Paste the error above and I'll write a loader for the raw files.[/]")

console.print(Panel.fit("[bold green]All done![/]", border_style="green"))

╭────────────────────────────────── 🚀 HF Downloader ───────────────────────────────────╮
│ Dataset: stindardlogic/structured-output-sft-100k                                     │
│ Cache:   C:\hf_cache                                                                  │
│ Save to: C:\Users\krish\OneDrive\Desktop\all-about-slms\structure-output-slm\dataset4 │
╰───────────────────────────────────────────────────────────────────────────────────────╯

────────────────────────────────────────────── Step 1: Repo contents ──────────────────────────────────────────────

                    3 files                     
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━┓
┃ File                             ┃ Size (MB) ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━┩
│ .gitattributes                   │       0.0 │
│ README.md                        │       0.0 │
│ structured-output-sft-100k.jsonl │      94.5 │
└──────────────────────────────────┴───────────┘

────────────────────────────────────────────── Step 2: load_dataset ───────────────────────────────────────────────

Generating train split: 100%|██████████| 100000/100000 [00:00<00:00, 136854.89 examples/s]


✔ Loaded in 10.1s

... (5 outputs left out)

───────────────────────────────────────────────── Step 3: Saving ──────────────────────────────────────────────────

Saving the dataset (1/1 shards): 100%|██████████| 100000/100000 [00:00<00:00, 429361.09 examples/s]


✔ Saved in 0.2s (79.4 MB) → C:\Users\krish\OneDrive\Desktop\all-about-slms\structure-output-slm\dataset4\hf_dataset

╭───────────╮
│ All done! │
╰───────────╯

In [10]:
import os, time, json, shutil
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"   # Windows noise

from pathlib import Path
from collections import Counter
from huggingface_hub import snapshot_download
from rich.console import Console
from rich.panel import Panel
from rich.table import Table
from rich.rule import Rule
from rich.syntax import Syntax
from rich.progress import Progress, SpinnerColumn, TextColumn, TimeElapsedColumn

console = Console()

REPO = "mdonigian/synthetic-structured-output-dataset"
CACHE_DIR = r"C:\hf_cache"
SAVE_ROOT = Path(r"C:\Users\krish\OneDrive\Desktop\all-about-slms\structure-output-slm\dataset5")
SAVE_ROOT.mkdir(parents=True, exist_ok=True)

console.print(Panel.fit(
    f"[bold cyan]Dataset:[/] {REPO}\n"
    f"[bold cyan]Cache:[/]   {CACHE_DIR}\n"
    f"[bold cyan]Save to:[/] {SAVE_ROOT}",
    title="[bold magenta]🚀 HF Downloader[/]", border_style="magenta",
))

# 1. Download raw files
console.print(Rule("[bold yellow]Step 1: Downloading repo files[/]"))
t0 = time.time()
local_dir = Path(snapshot_download(
    REPO, repo_type="dataset", cache_dir=CACHE_DIR,
    allow_patterns=["*.jsonl", "*.json", "README.md"],
))
console.print(f"[bold green]✔ Downloaded in {time.time() - t0:.1f}s[/] → [dim]{local_dir}[/]")

# 2. Copy to save folder
console.print(Rule("[bold yellow]Step 2: Copying files[/]"))
jsonl_files = sorted(local_dir.rglob("*.jsonl"))
other_files = [f for f in local_dir.rglob("*.json")] + list(local_dir.rglob("README.md"))
if not jsonl_files:
    console.print("[bold red]✘ No .jsonl files found. Check the repo layout.[/]")
    raise SystemExit(1)

ftable = Table(title="Files", header_style="bold cyan", border_style="blue")
ftable.add_column("File", style="bold green")
ftable.add_column("Size (MB)", justify="right", style="yellow")
copied = {}
for f in jsonl_files + other_files:
    dst = SAVE_ROOT / f.name
    shutil.copy2(f, dst)
    copied[f.name] = dst
    ftable.add_row(f.name, f"{dst.stat().st_size / 1024**2:,.2f}")
console.print(ftable)

# 3. Scan each jsonl
console.print(Rule("[bold magenta]Step 3: Scanning files[/]"))
summary = Table(title="Summary", header_style="bold cyan", border_style="blue")
summary.add_column("File", style="bold green")
summary.add_column("Rows", justify="right", style="yellow")
summary.add_column("Bad", justify="right", style="red")
summary.add_column("Fields", style="white")

grand_total = 0
for f in jsonl_files:
    path = copied[f.name]
    keys, errors, sources = Counter(), Counter(), Counter()
    total, bad, sample = 0, 0, None

    with Progress(
        SpinnerColumn(style="magenta"),
        TextColumn(f"[cyan]{f.name}[/] [yellow]{{task.completed:,}}[/] rows"),
        TimeElapsedColumn(),
        console=console, transient=True,
    ) as prog:
        task = prog.add_task("scan", total=None)
        with open(path, encoding="utf-8") as fh:
            for line in fh:
                try:
                    r = json.loads(line)
                except json.JSONDecodeError:
                    bad += 1
                    continue
                total += 1
                if sample is None:
                    sample = r
                keys.update(r.keys())
                if "error_type" in r:
                    errors[r["error_type"]] += 1
                if "source" in r:
                    sources[r["source"]] += 1
                prog.advance(task)

    grand_total += total
    summary.add_row(f.name, f"{total:,}", str(bad), ", ".join(keys))
    console.print(f"[bold green]✔ {f.name}[/]: [yellow]{total:,}[/] rows")

    for title, counter in (("error_type", errors), ("source", sources)):
        if counter:
            t = Table(title=f"{f.name} → {title}", header_style="bold magenta", border_style="magenta")
            t.add_column("Value", style="bold green")
            t.add_column("Rows", justify="right", style="yellow")
            t.add_column("%", justify="right", style="dim")
            for k, v in counter.most_common():
                t.add_row(str(k), f"{v:,}", f"{v / total * 100:.1f}")
            console.print(t)

    console.print(Panel(
        Syntax(json.dumps(sample, indent=2, ensure_ascii=False)[:1200], "json", theme="monokai", word_wrap=True),
        title=f"[bold cyan]Sample row: {f.name}[/]", border_style="cyan",
    ))

console.print(Rule("[bold green]Finished[/]"))
console.print(summary)
console.print(f"[bold cyan]Total rows across files:[/] [yellow]{grand_total:,}[/]  [dim](README says 32,914)[/]")

# 4. QA report
qa = copied.get("synthetic_qa_report.json")
if qa:
    console.print(Rule("[bold yellow]QA report[/]"))
    txt = qa.read_text(encoding="utf-8")
    console.print(Syntax(txt[:2500], "json", theme="monokai", word_wrap=True))

console.print(Panel.fit(f"[bold green]All done![/]\n{SAVE_ROOT}", border_style="green"))

╭────────────────────────────────── 🚀 HF Downloader ───────────────────────────────────╮
│ Dataset: mdonigian/synthetic-structured-output-dataset                                │
│ Cache:   C:\hf_cache                                                                  │
│ Save to: C:\Users\krish\OneDrive\Desktop\all-about-slms\structure-output-slm\dataset5 │
╰───────────────────────────────────────────────────────────────────────────────────────╯

───────────────────────────────────────── Step 1: Downloading repo files ──────────────────────────────────────────

Fetching 6 files: 100%|██████████| 6/6 [00:08<00:00,  1.34s/it]


✔ Downloaded in 8.6s → 
C:\hf_cache\datasets--mdonigian--synthetic-structured-output-dataset\snapshots\06fbd8ea2733fcef3120d9d5932f1026c666
ac05

────────────────────────────────────────────── Step 2: Copying files ──────────────────────────────────────────────

                    Files                     
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━┓
┃ File                           ┃ Size (MB) ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━┩
│ dpo_function_calling.jsonl     │      2.23 │
│ dpo_structured_output.jsonl    │    104.02 │
│ sft_synthetic_extraction.jsonl │     14.64 │
│ sft_synthetic_json.jsonl       │     31.36 │
│ synthetic_qa_report.json       │      0.00 │
│ README.md                      │      0.00 │
└────────────────────────────────┴───────────┘

... (17 outputs left out)

Total rows across files: 32,914  (README says 32,914)

──────────────────────────────────────────────────── QA report ────────────────────────────────────────────────────

# Synthetic Data QA Report                                                                                         
                                                                                                                   
## Counts                                                                                                          
                                                                                                                   
| Dataset | Count | Target |                                                                                       
|---------|-------|--------|                                                                                       
| SFT JSON Schema Compliance | 9123 | 9500 |                                                                       
| SFT Structured Extraction | 4663 | 4750 |                                                                        
| DPO Structured Output | 14128 | 15000 |                                                                          
| DPO Function Calling | 5000 | 5000 |                                                                             
| **Total** | **32914** | |                                                                                        
                                                                                                                   
## Validation Pass Rates                                                                                           
                                                                                                                   
- SFT JSON: 9123/9123 (100.0%)                                                                                     
- Extraction: 4663/4663 (100.0%)                                                                                   
- DPO Structured: 14128/14128 (100.0%)                                                                             
- DPO Function Calling: 2876/5000 (57.5%)                                                                          
                                                                                                                   
## Contamination Check                                                                                             
                                                                                                                   
Skipped (no eval schemas provi

╭──────────────────────────────────────────────────────────────────────────────╮
│ All done!                                                                    │
│ C:\Users\krish\OneDrive\Desktop\all-about-slms\structure-output-slm\dataset5 │
╰──────────────────────────────────────────────────────────────────────────────╯

In [15]:
import os, time, shutil, traceback
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"   # Windows noise

from pathlib import Path
from datasets import load_dataset
from huggingface_hub import HfApi, snapshot_download
from rich.console import Console
from rich.panel import Panel
from rich.table import Table
from rich.rule import Rule

console = Console()

REPO = "interfaze-ai/sob"
CONFIG = "default"
CACHE_DIR = r"C:\hf_cache"
SAVE_ROOT = Path(r"C:\Users\krish\OneDrive\Desktop\all-about-slms\structure-output-slm\dataset6")
SAVE_ROOT.mkdir(parents=True, exist_ok=True)

console.print(Panel.fit(
    f"[bold cyan]Dataset:[/] {REPO}  [dim](config: {CONFIG})[/]\n"
    f"[bold cyan]Cache:[/]   {CACHE_DIR}\n"
    f"[bold cyan]Save to:[/] {SAVE_ROOT}",
    title="[bold magenta]🚀 HF Downloader[/]", border_style="magenta",
))

# 1. Peek at the repo files
console.print(Rule("[bold yellow]Step 1: Repo contents[/]"))
info = HfApi().dataset_info(REPO, files_metadata=True)
ftable = Table(title=f"{len(info.siblings)} files", header_style="bold cyan", border_style="blue")
ftable.add_column("File", style="bold green")
ftable.add_column("Size (MB)", justify="right", style="yellow")
for s in sorted(info.siblings, key=lambda s: s.rfilename):
    ftable.add_row(s.rfilename, f"{(s.size or 0) / 1024**2:,.2f}")
console.print(ftable)

# 2. Try the normal load
console.print(Rule("[bold yellow]Step 2: load_dataset[/]"))
t0 = time.time()
try:
    ds = load_dataset(REPO, CONFIG, cache_dir=CACHE_DIR)
    console.print(f"[bold green]✔ Loaded in {time.time() - t0:.1f}s[/]")

    table = Table(title="Dataset Summary", header_style="bold cyan", border_style="blue")
    table.add_column("Split", style="bold green")
    table.add_column("Rows", justify="right", style="yellow")
    table.add_column("Columns", style="white")
    for split, d in ds.items():
        table.add_row(split, f"{d.num_rows:,}", ", ".join(d.column_names))
    console.print(table)

    # sample row
    first = next(iter(ds.values()))
    console.print(Rule("[bold magenta]Sample row[/]"))
    for k, v in first[0].items():
        console.print(f"[bold cyan]{k}[/]: [white]{str(v)[:300]}[/]")

    # 3. Save
    console.print(Rule("[bold yellow]Step 3: Saving[/]"))
    t0 = time.time()
    out = SAVE_ROOT / "hf_dataset"
    ds.save_to_disk(str(out))
    size_mb = sum(f.stat().st_size for f in out.rglob("*") if f.is_file()) / 1024**2
    console.print(f"[bold green]✔ Saved in {time.time() - t0:.1f}s[/] [dim]({size_mb:,.1f} MB)[/] → {out}")

except Exception as e:
    # Fallback: grab raw files so you still have the data
    console.print(f"[bold red]✘ load_dataset failed:[/] {str(e)[:300]}")
    console.print(traceback.format_exc(), style="dim red")
    console.print(Rule("[bold yellow]Fallback: downloading raw files[/]"))

    t0 = time.time()
    local_dir = Path(snapshot_download(REPO, repo_type="dataset", cache_dir=CACHE_DIR))
    raw_out = SAVE_ROOT / "raw"
    shutil.copytree(local_dir, raw_out, dirs_exist_ok=True)
    console.print(f"[bold green]✔ Raw files copied in {time.time() - t0:.1f}s[/] → {raw_out}")
    console.print("[yellow]Paste the error above and the file table from Step 1, and I'll write a loader for the raw files.[/]")

console.print(Panel.fit("[bold green]All done![/]", border_style="green"))

╭────────────────────────────────── 🚀 HF Downloader ───────────────────────────────────╮
│ Dataset: interfaze-ai/sob  (config: default)                                          │
│ Cache:   C:\hf_cache                                                                  │
│ Save to: C:\Users\krish\OneDrive\Desktop\all-about-slms\structure-output-slm\dataset6 │
╰───────────────────────────────────────────────────────────────────────────────────────╯

────────────────────────────────────────────── Step 1: Repo contents ──────────────────────────────────────────────

                       8 files                        
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━┓
┃ File                                   ┃ Size (MB) ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━┩
│ .gitattributes                         │      0.00 │
│ LICENSE                                │      0.00 │
│ README.md                              │      0.01 │
│ audio/train-00000-of-00001.parquet     │      2.51 │
│ data/test-00000-of-00001.parquet       │     18.80 │
│ data/train-00000-of-00001.parquet      │     66.39 │
│ data/validation-00000-of-00001.parquet │      7.44 │
│ image/train-00000-of-00001.parquet     │      0.92 │
└────────────────────────────────────────┴───────────┘

────────────────────────────────────────────── Step 2: load_dataset ───────────────────────────────────────────────

✔ Loaded in 2.4s

                                                  Dataset Summary                                                  
┏━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Split      ┃   Rows ┃ Columns                                                                                   ┃
┡━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ train      │ 17,699 │ record_id, question_type, question_difficulty, schema_complexity, source_id,              │
│            │        │ source_dataset, source_answer, pipeline_version, context, question, json_schema,          │
│            │        │ candidate_response, ground_truth, validated_output                                        │
│ validation │  1,966 │ record_id, question_type, question_difficulty, schema_complexity, source_id,              │
│            │        │ source_dataset, source_answer, pipeline_version, context, question, json_schema,          │
│            │        │ candidate_response, ground_truth, validated_output                                        │
│ test       │  5,000 │ record_id, question_type, question_difficulty, schema_complexity, source_id,              │
│            │        │ source_dataset, source_answer, pipeline_version, context, question, json_schema,          │
│            │        │ candidate_response, ground_truth, validated_output                                        │
└────────────┴────────┴───────────────────────────────────────────────────────────────────────────────────────────┘

... (15 outputs left out)

───────────────────────────────────────────────── Step 3: Saving ──────────────────────────────────────────────────

Saving the dataset (1/1 shards): 100%|██████████| 5000/5000 [00:00<00:00, 97130.84 examples/s] 


✔ Saved in 0.3s (172.3 MB) → 
C:\Users\krish\OneDrive\Desktop\all-about-slms\structure-output-slm\dataset6\hf_dataset

╭───────────╮
│ All done! │
╰───────────╯

In [12]:
import os, time, shutil, traceback
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"   # Windows noise

from pathlib import Path
from datasets import load_dataset
from huggingface_hub import HfApi, snapshot_download
from rich.console import Console
from rich.panel import Panel
from rich.table import Table
from rich.rule import Rule

console = Console()

REPO = "areias/conll2003-generative"
CACHE_DIR = r"C:\hf_cache"
SAVE_ROOT = Path(r"C:\Users\krish\OneDrive\Desktop\all-about-slms\structure-output-slm\dataset7")
SAVE_ROOT.mkdir(parents=True, exist_ok=True)

console.print(Panel.fit(
    f"[bold cyan]Dataset:[/] {REPO}\n"
    f"[bold cyan]Cache:[/]   {CACHE_DIR}\n"
    f"[bold cyan]Save to:[/] {SAVE_ROOT}",
    title="[bold magenta]🚀 HF Downloader[/]", border_style="magenta",
))

# 1. Peek at the repo files
console.print(Rule("[bold yellow]Step 1: Repo contents[/]"))
info = HfApi().dataset_info(REPO, files_metadata=True)
ftable = Table(title=f"{len(info.siblings)} files", header_style="bold cyan", border_style="blue")
ftable.add_column("File", style="bold green")
ftable.add_column("Size (MB)", justify="right", style="yellow")
for s in sorted(info.siblings, key=lambda s: s.rfilename):
    ftable.add_row(s.rfilename, f"{(s.size or 0) / 1024**2:,.2f}")
console.print(ftable)

# 2. Try the normal load
console.print(Rule("[bold yellow]Step 2: load_dataset[/]"))
t0 = time.time()
try:
    ds = load_dataset(REPO, cache_dir=CACHE_DIR)
    console.print(f"[bold green]✔ Loaded in {time.time() - t0:.1f}s[/]")

    table = Table(title="Dataset Summary", header_style="bold cyan", border_style="blue")
    table.add_column("Split", style="bold green")
    table.add_column("Rows", justify="right", style="yellow")
    table.add_column("Columns", style="white")
    for split, d in ds.items():
        table.add_row(split, f"{d.num_rows:,}", ", ".join(d.column_names))
    console.print(table)

    # sample rows
    first = next(iter(ds.values()))
    console.print(Rule("[bold magenta]Sample row[/]"))
    for k, v in first[0].items():
        console.print(f"[bold cyan]{k}[/]: [white]{str(v)[:300]}[/]")

    # 3. Save
    console.print(Rule("[bold yellow]Step 3: Saving[/]"))
    t0 = time.time()
    out = SAVE_ROOT / "hf_dataset"
    ds.save_to_disk(str(out))
    size_mb = sum(f.stat().st_size for f in out.rglob("*") if f.is_file()) / 1024**2
    console.print(f"[bold green]✔ Saved in {time.time() - t0:.1f}s[/] [dim]({size_mb:,.1f} MB)[/] → {out}")

except Exception as e:
    # Fallback: grab raw files so you still have the data
    console.print(f"[bold red]✘ load_dataset failed:[/] {str(e)[:300]}")
    console.print(traceback.format_exc(), style="dim red")
    console.print(Rule("[bold yellow]Fallback: downloading raw files[/]"))

    t0 = time.time()
    local_dir = Path(snapshot_download(REPO, repo_type="dataset", cache_dir=CACHE_DIR))
    raw_out = SAVE_ROOT / "raw"
    shutil.copytree(local_dir, raw_out, dirs_exist_ok=True)
    console.print(f"[bold green]✔ Raw files copied in {time.time() - t0:.1f}s[/] → {raw_out}")
    console.print("[yellow]Paste the error above and the file table from Step 1, and I'll write a loader for the raw files.[/]")

console.print(Panel.fit("[bold green]All done![/]", border_style="green"))

╭────────────────────────────────── 🚀 HF Downloader ───────────────────────────────────╮
│ Dataset: areias/conll2003-generative                                                  │
│ Cache:   C:\hf_cache                                                                  │
│ Save to: C:\Users\krish\OneDrive\Desktop\all-about-slms\structure-output-slm\dataset7 │
╰───────────────────────────────────────────────────────────────────────────────────────╯

────────────────────────────────────────────── Step 1: Repo contents ──────────────────────────────────────────────

                                7 files                                
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━┓
┃ File                                                    ┃ Size (MB) ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━┩
│ .gitattributes                                          │      0.00 │
│ LICENSE                                                 │      0.00 │
│ README.md                                               │      0.00 │
│ data/test-00000-of-00001-d1ac0aba9982de6b.parquet       │      0.50 │
│ data/train-00000-of-00001-b96d203e08a6b884.parquet      │      2.16 │
│ data/validation-00000-of-00001-03bb0539ee1a88da.parquet │      0.55 │
│ preprocess_data.py                                      │      0.00 │
└─────────────────────────────────────────────────────────┴───────────┘

────────────────────────────────────────────── Step 2: load_dataset ───────────────────────────────────────────────

Generating test split: 100%|██████████| 3453/3453 [00:00<00:00, 128692.56 examples/s]


✔ Loaded in 11.3s

... (10 outputs left out)

───────────────────────────────────────────────── Step 3: Saving ──────────────────────────────────────────────────

Saving the dataset (1/1 shards): 100%|██████████| 3453/3453 [00:00<00:00, 121825.09 examples/s]


✔ Saved in 0.2s (13.9 MB) → C:\Users\krish\OneDrive\Desktop\all-about-slms\structure-output-slm\dataset7\hf_dataset

╭───────────╮
│ All done! │
╰───────────╯

In [16]:
import os, time, shutil, traceback
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"   # Windows noise

from pathlib import Path
from datasets import load_dataset
from huggingface_hub import HfApi, snapshot_download
from rich.console import Console
from rich.panel import Panel
from rich.table import Table
from rich.rule import Rule

console = Console()

REPO = "buthaya/sroie"
CACHE_DIR = r"C:\hf_cache"
SAVE_ROOT = Path(r"C:\Users\krish\OneDrive\Desktop\all-about-slms\structure-output-slm\dataset8")
SAVE_ROOT.mkdir(parents=True, exist_ok=True)

console.print(Panel.fit(
    f"[bold cyan]Dataset:[/] {REPO}\n"
    f"[bold cyan]Cache:[/]   {CACHE_DIR}\n"
    f"[bold cyan]Save to:[/] {SAVE_ROOT}",
    title="[bold magenta]🚀 HF Downloader[/]", border_style="magenta",
))

# 0. Auth check (repo may be gated)
console.print(Rule("[bold yellow]Step 0: Auth check[/]"))
try:
    who = HfApi().whoami()
    console.print(f"[bold green]✔ Logged in as:[/] [cyan]{who['name']}[/]")
except Exception:
    console.print("[bold red]✘ Not logged in.[/] Run [bold]huggingface-cli login[/] in a terminal "
                  "(or [bold]from huggingface_hub import login; login()[/]), "
                  "then accept the dataset terms on its HF page if it's gated.")
    raise SystemExit(1)

# 1. Peek at the repo files
console.print(Rule("[bold yellow]Step 1: Repo contents[/]"))
info = HfApi().dataset_info(REPO, files_metadata=True)
ftable = Table(title=f"{len(info.siblings)} files", header_style="bold cyan", border_style="blue")
ftable.add_column("File", style="bold green")
ftable.add_column("Size (MB)", justify="right", style="yellow")
for s in sorted(info.siblings, key=lambda s: s.rfilename):
    ftable.add_row(s.rfilename, f"{(s.size or 0) / 1024**2:,.2f}")
console.print(ftable)

# 2. Try the normal load
console.print(Rule("[bold yellow]Step 2: load_dataset[/]"))
t0 = time.time()
try:
    ds = load_dataset(REPO, cache_dir=CACHE_DIR)
    console.print(f"[bold green]✔ Loaded in {time.time() - t0:.1f}s[/]")

    table = Table(title="Dataset Summary", header_style="bold cyan", border_style="blue")
    table.add_column("Split", style="bold green")
    table.add_column("Rows", justify="right", style="yellow")
    table.add_column("Columns", style="white")
    for split, d in ds.items():
        table.add_row(split, f"{d.num_rows:,}", ", ".join(d.column_names))
    console.print(table)

    # sample row (images shown as type + size instead of raw bytes)
    first = next(iter(ds.values()))
    console.print(Rule("[bold magenta]Sample row[/]"))
    for k, v in first[0].items():
        if hasattr(v, "size") and hasattr(v, "mode"):
            console.print(f"[bold cyan]{k}[/]: [white]<image {v.size[0]}x{v.size[1]} {v.mode}>[/]")
        else:
            console.print(f"[bold cyan]{k}[/]: [white]{str(v)[:300]}[/]")

    # 3. Save
    console.print(Rule("[bold yellow]Step 3: Saving[/]"))
    t0 = time.time()
    out = SAVE_ROOT / "hf_dataset"
    ds.save_to_disk(str(out))
    size_mb = sum(f.stat().st_size for f in out.rglob("*") if f.is_file()) / 1024**2
    console.print(f"[bold green]✔ Saved in {time.time() - t0:.1f}s[/] [dim]({size_mb:,.1f} MB)[/] → {out}")

except Exception as e:
    # Fallback: grab raw files so you still have the data
    console.print(f"[bold red]✘ load_dataset failed:[/] {str(e)[:300]}")
    console.print(traceback.format_exc(), style="dim red")
    console.print(Rule("[bold yellow]Fallback: downloading raw files[/]"))

    t0 = time.time()
    local_dir = Path(snapshot_download(REPO, repo_type="dataset", cache_dir=CACHE_DIR))
    raw_out = SAVE_ROOT / "raw"
    shutil.copytree(local_dir, raw_out, dirs_exist_ok=True)
    console.print(f"[bold green]✔ Raw files copied in {time.time() - t0:.1f}s[/] → {raw_out}")
    console.print("[yellow]Paste the error above and the file table from Step 1, and I'll write a loader for the raw files.[/]")

console.print(Panel.fit("[bold green]All done![/]", border_style="green"))

╭────────────────────────────────── 🚀 HF Downloader ───────────────────────────────────╮
│ Dataset: buthaya/sroie                                                                │
│ Cache:   C:\hf_cache                                                                  │
│ Save to: C:\Users\krish\OneDrive\Desktop\all-about-slms\structure-output-slm\dataset8 │
╰───────────────────────────────────────────────────────────────────────────────────────╯

─────────────────────────────────────────────── Step 0: Auth check ────────────────────────────────────────────────

✔ Logged in as: KParikh2005

────────────────────────────────────────────── Step 1: Repo contents ──────────────────────────────────────────────

                             983 files                              
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━┓
┃ File                                                 ┃ Size (MB) ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━┩
│ .gitattributes                                       │      0.00 │
│ README.md                                            │      0.00 │
│ label_list.json                                      │      0.00 │
│ test/annotations/X00016469670.json                   │      0.00 │
│ test/annotations/X00016469671.json                   │      0.00 │
│ test/annotations/X51005200931.json                   │      0.01 │
│ test/annotations/X51005230605.json                   │      0.00 │
│ test/annotations/X51005230616.json                   │      0.00 │
│ test/annotations/X51005230621.json                   │      0.00 │
│ test/annotations/X51005230648.json                   │      0.00 │
│ test/annotations/X51005230657.json                   │      0.00 │
│ test/annotations/X51005230659.json                   │      0.00 │
│ test/annotations/X51005268275.json                   │      0.01 │
│ test/annotations/X51005268408.json                   │      0.00 │
│ test/annotations/X51005288570.json                   │      0.00 │
│ test/annotations/X51005301666.json                   │      0.01 │
│ test/annotations/X51005337867.json                   │      0.00 │
│ test/annotations/X51005337877.json                   │      0.00 │
│ test/annotations/X51005361906.json                   │      0.01 │
│ test/annotations/X51005361908.json                   │      0.00 │
│ test/annotations/X51005361912.json                   │      0.00 │
│ test/annotations/X51005361923.json                   │      0.00 │
│ test/annotations/X51005365187.json                   │      0.01 │
│ test/annotations/X51005433518.json                   │      0.00 │
│ test/annotations/X51005433543.json                   │      0.00 │
│ test/annotations/X51005433548.json                   │      0.00 │
│ test/annotations/X51005433556.json                   │      0.00 │
│ test/annotations/X51005442322.json                   │      0.00 │
│ test/annotations/X51005442334.json                   │      0.00 │
│ test/annotations/X51005442343.json                   │      0.00 │
│ test/annotations/X51005442366.json                   │      0.00 │
│ test/annotations/X51005442375.json                   │      0.00 │
│ test/annotations/X51005442382.json                   │      0.00 │
│ test/annotations/X51005442388.json                   │      0.00 │
│ test/annotations/X51005444040.json                   │      0.00 │
│ test/annotations/X51005444041.json                   │      0.00 │
│ test/annotations/X51005444044.json                   │      0.00 │
│ test/annotations/X51005444046.json                   │      0.00 │
│ test/annotations/X51005447841.json                   │      0.00 │
│ test/annotations/X51005447842.json                   │      0.00 │
│ test/annotations/X51005447844.json                   │      0.00 │
│ test/annotations/X51005447851.json                   │      0.00 │
│ test/annotations/X51005447859.json                   │      0.00 │
│ test/annotations/X51005568855.json                   │      0.01 │
│ test/annotations/X51005568866.json                   │      0.00 │
│ test/annotations/X51005568885.json                   │      0.00 │
│ test/annotations/X51005568887.json                   │      0.00 │
│ test/annotations/X51005568889.json                   │      0.00 │
│ test/annotations/X51005568890.json                   │      0.00 │
│ test/annotations/X51005568892.json                   │      0.00 │
│ test/annotations/X51005568894.json                   │      0.01 │
│ test/annotations/X51005568895.json                   │      0.00 │
│ test/annotations/X51005577191.json                   │      0.00 │
│ test/annotations/X51005582745.json                   │      0.00 

────────────────────────────────────────────── Step 2: load_dataset ───────────────────────────────────────────────

... (12 outputs left out)

───────────────────────────────────────────────── Step 3: Saving ──────────────────────────────────────────────────

Saving the dataset (1/1 shards): 100%|██████████| 347/347 [00:00<00:00, 13137.01 examples/s]


✔ Saved in 0.1s (5.0 MB) → C:\Users\krish\OneDrive\Desktop\all-about-slms\structure-output-slm\dataset8\hf_dataset

╭───────────╮
│ All done! │
╰───────────╯

In [18]:
import os, time, traceback
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"   # Windows noise

from pathlib import Path
from collections import defaultdict
from datasets import load_dataset
from huggingface_hub import HfApi, snapshot_download
from rich.console import Console
from rich.panel import Panel
from rich.table import Table
from rich.rule import Rule

console = Console()

REPO = "GEM/schema_guided_dialog"
REV = "refs/convert/parquet"
CACHE_DIR = r"C:\hf_cache"
SAVE_ROOT = Path(r"C:\Users\krish\OneDrive\Desktop\all-about-slms\structure-output-slm\dataset9")
SAVE_ROOT.mkdir(parents=True, exist_ok=True)

console.print(Panel.fit(
    f"[bold cyan]Dataset:[/]  {REPO}\n"
    f"[bold cyan]Revision:[/] {REV}\n"
    f"[bold cyan]Cache:[/]    {CACHE_DIR}\n"
    f"[bold cyan]Save to:[/]  {SAVE_ROOT}",
    title="[bold magenta]🚀 HF Downloader (parquet branch)[/]", border_style="magenta",
))

# 1. List parquet files on the converted branch
console.print(Rule("[bold yellow]Step 1: Listing parquet files[/]"))
try:
    all_files = HfApi().list_repo_files(REPO, repo_type="dataset", revision=REV)
except Exception as e:
    console.print(f"[bold red]✘ Could not list {REV}:[/] {str(e)[:300]}")
    console.print("[yellow]The auto-converted branch isn't available for this repo. "
                  "Tell me and I'll write a downloader for the original Google dstc8 data instead.[/]")
    raise SystemExit(1)

pq_files = [f for f in all_files if f.endswith(".parquet")]
if not pq_files:
    console.print("[bold red]✘ No parquet files on that branch.[/]")
    raise SystemExit(1)

listing = Table(title=f"{len(pq_files)} parquet files", header_style="bold cyan", border_style="blue")
listing.add_column("Path", style="bold green")
for f in sorted(pq_files):
    listing.add_row(f)
console.print(listing)

# 2. Download them
console.print(Rule("[bold yellow]Step 2: Downloading[/]"))
t0 = time.time()
local_dir = Path(snapshot_download(
    REPO, repo_type="dataset", revision=REV, cache_dir=CACHE_DIR,
    allow_patterns=["*.parquet"],
))
console.print(f"[bold green]✔ Downloaded in {time.time() - t0:.1f}s[/] → [dim]{local_dir}[/]")

# 3. Group by config / split  (layout: <config>/<split>/<shard>.parquet)
cfgs = defaultdict(lambda: defaultdict(list))
for f in sorted(local_dir.rglob("*.parquet")):
    parts = f.relative_to(local_dir).parts
    config = parts[0] if len(parts) >= 3 else "default"
    split = parts[-2] if len(parts) >= 2 else "train"
    cfgs[config][split].append(str(f))

# 4. Load + save each config
summary = Table(title="Results", header_style="bold cyan", border_style="blue")
summary.add_column("Config", style="bold green")
summary.add_column("Split", style="cyan")
summary.add_column("Rows", justify="right", style="yellow")
summary.add_column("Status")

for config, splits in cfgs.items():
    console.print(Rule(f"[bold magenta]Config: {config}[/]"))
    try:
        t0 = time.time()
        ds = load_dataset("parquet", data_files=dict(splits), cache_dir=CACHE_DIR)

        for split, d in ds.items():
            console.print(f"  [cyan]{split}[/]: [yellow]{d.num_rows:,}[/] rows | cols: [white]{d.column_names}[/]")
            summary.add_row(config, split, f"{d.num_rows:,}", "[green]✔ OK[/]")

        # sample row from first split
        first = next(iter(ds.values()))
        console.print("[bold magenta]Sample row:[/]")
        for k, v in first[0].items():
            console.print(f"  [bold cyan]{k}[/]: [white]{str(v)[:300]}[/]")

        out = SAVE_ROOT / config
        ds.save_to_disk(str(out))
        size_mb = sum(p.stat().st_size for p in out.rglob("*") if p.is_file()) / 1024**2
        console.print(f"[bold green]✔ Saved in {time.time() - t0:.1f}s[/] [dim]({size_mb:,.1f} MB)[/] → {out}")
    except Exception as e:
        console.print(f"[bold red]✘ {config} failed:[/] {e}")
        console.print(traceback.format_exc(), style="dim red")
        summary.add_row(config, "-", "-", "[red]✘ FAILED[/]")

console.print(Rule("[bold green]Finished[/]"))
console.print(summary)
console.print(Panel.fit(f"[bold green]All done![/]\n{SAVE_ROOT}", border_style="green"))

╭────────────────────────── 🚀 HF Downloader (parquet branch) ───────────────────────────╮
│ Dataset:  GEM/schema_guided_dialog                                                     │
│ Revision: refs/convert/parquet                                                         │
│ Cache:    C:\hf_cache                                                                  │
│ Save to:  C:\Users\krish\OneDrive\Desktop\all-about-slms\structure-output-slm\dataset9 │
╰────────────────────────────────────────────────────────────────────────────────────────╯

────────────────────────────────────────── Step 1: Listing parquet files ──────────────────────────────────────────

                   10 parquet files                    
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Path                                                ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ default/challenge_test_backtranslation/0000.parquet │
│ default/challenge_test_bfp02/0000.parquet           │
│ default/challenge_test_bfp05/0000.parquet           │
│ default/challenge_test_nopunc/0000.parquet          │
│ default/challenge_test_scramble/0000.parquet        │
│ default/challenge_train_sample/0000.parquet         │
│ default/challenge_validation_sample/0000.parquet    │
│ default/test/0000.parquet                           │
│ default/train/0000.parquet                          │
│ default/validation/0000.parquet                     │
└─────────────────────────────────────────────────────┘

─────────────────────────────────────────────── Step 2: Downloading ───────────────────────────────────────────────

Fetching 10 files: 100%|██████████| 10/10 [00:10<00:00,  1.05s/it]


✔ Downloaded in 11.0s → 
C:\hf_cache\datasets--GEM--schema_guided_dialog\snapshots\5f16d34e3196a489d5e6fc538db752b3539c086e

... (25 outputs left out)

✔ Saved in 3.1s (207.7 MB) → C:\Users\krish\OneDrive\Desktop\all-about-slms\structure-output-slm\dataset9\default

──────────────────────────────────────────────────── Finished ─────────────────────────────────────────────────────

                            Results                            
┏━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━┳━━━━━━━━┓
┃ Config  ┃ Split                          ┃    Rows ┃ Status ┃
┡━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━╇━━━━━━━━┩
│ default │ challenge_test_backtranslation │     500 │ ✔ OK   │
│ default │ challenge_test_bfp02           │     500 │ ✔ OK   │
│ default │ challenge_test_bfp05           │     500 │ ✔ OK   │
│ default │ challenge_test_nopunc          │     500 │ ✔ OK   │
│ default │ challenge_test_scramble        │     500 │ ✔ OK   │
│ default │ challenge_train_sample         │     500 │ ✔ OK   │
│ default │ challenge_validation_sample    │     500 │ ✔ OK   │
│ default │ test                           │  10,000 │ ✔ OK   │
│ default │ train                          │ 164,982 │ ✔ OK   │
│ default │ validation                     │  10,000 │ ✔ OK   │
└─────────┴────────────────────────────────┴─────────┴────────┘

╭──────────────────────────────────────────────────────────────────────────────╮
│ All done!                                                                    │
│ C:\Users\krish\OneDrive\Desktop\all-about-slms\structure-output-slm\dataset9 │
╰──────────────────────────────────────────────────────────────────────────────╯

In [19]:
import os
import json
import csv
from pathlib import Path
from collections import Counter

import pyarrow.parquet as pq

ROOT = Path(".")
DATASET_FOLDERS = [
    f"dataset{i}" for i in range(2, 10)
]

SUPPORTED = {
    ".jsonl", ".json", ".parquet",
    ".csv", ".txt", ".yaml", ".yml"
}


def inspect_jsonl(path):
    count = 0
    samples = []
    errors = 0
    keys = Counter()

    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            if not line.strip():
                continue

            try:
                record = json.loads(line)
                count += 1

                if isinstance(record, dict):
                    keys.update(record.keys())

                if len(samples) < 3:
                    samples.append(record)

            except Exception:
                errors += 1

    return {
        "rows": count,
        "errors": errors,
        "common_keys": keys.most_common(20),
        "samples": samples
    }


def inspect_parquet(path):
    pf = pq.ParquetFile(path)

    samples = pf.read_row_group(
        0,
        columns=pf.schema_arrow.names
    ).slice(0, 3).to_pylist() if pf.num_row_groups else []

    return {
        "rows": pf.metadata.num_rows,
        "columns": pf.schema_arrow.names,
        "schema": str(pf.schema_arrow),
        "samples": samples
    }


def inspect_csv(path):
    with open(path, "r", encoding="utf-8-sig",
              newline="") as f:
        reader = csv.DictReader(f)
        samples = []

        for row in reader:
            if len(samples) < 3:
                samples.append(row)

        return {
            "columns": reader.fieldnames,
            "samples": samples
        }


def audit_file(path):
    result = {
        "file": str(path),
        "size_mb": round(path.stat().st_size / 1024**2, 2),
        "extension": path.suffix.lower()
    }

    try:
        if path.suffix.lower() == ".jsonl":
            result.update(inspect_jsonl(path))

        elif path.suffix.lower() == ".parquet":
            result.update(inspect_parquet(path))

        elif path.suffix.lower() == ".csv":
            result.update(inspect_csv(path))

        elif path.suffix.lower() == ".json":
            with open(path, encoding="utf-8") as f:
                data = json.load(f)

            result["type"] = type(data).__name__
            result["samples"] = (
                data[:3] if isinstance(data, list)
                else data
            )

        elif path.suffix.lower() in {".yaml", ".yml"}:
            result["note"] = "YAML file - inspect separately"

        elif path.suffix.lower() == ".txt":
            with open(path, encoding="utf-8") as f:
                result["preview"] = f.read(1000)

    except Exception as e:
        result["error"] = str(e)

    return result


def main():
    report = {}

    for folder_name in DATASET_FOLDERS:
        folder = ROOT / folder_name

        if not folder.exists():
            report[folder_name] = {"status": "MISSING"}
            continue

        print(f"\nAuditing {folder_name}...")

        files = [
            p for p in folder.rglob("*")
            if p.is_file()
            and p.suffix.lower() in SUPPORTED
        ]

        report[folder_name] = {
            "status": "FOUND",
            "total_files": len(files),
            "files": [
                audit_file(path) for path in files
            ]
        }

        print(f"Found {len(files)} data files")

    output = ROOT / "dataset_audit.json"

    with open(output, "w", encoding="utf-8") as f:
        json.dump(report, f, indent=2, ensure_ascii=False,
                  default=str)

    print("\nAudit complete!")
    print(f"Report saved to: {output}")


if __name__ == "__main__":
    main()



Auditing dataset2...
Found 9 data files

Auditing dataset3...
Found 3 data files

Auditing dataset4...
Found 3 data files

Auditing dataset5...
Found 5 data files

Auditing dataset6...
Found 7 data files

Auditing dataset7...
Found 7 data files

Auditing dataset8...
Found 5 data files

Auditing dataset9...
Found 28 data files

Audit complete!
Report saved to: dataset_audit.json


In [20]:
from pathlib import Path
from datasets import load_from_disk, DatasetDict
import json

ROOT = Path(".")

DATASETS = {
    "nemotron_direct": "dataset2/direct_generation",
    "nemotron_diversified": "dataset2/diversified_tasks",
    "nemotron_tool_extraction": "dataset2/tool_calling_extraction",
    "emulated_pool": "dataset3/hf_dataset",
    "structured_sft": "dataset4/hf_dataset",
    "sob": "dataset6/hf_dataset",
    "conll": "dataset7/hf_dataset",
    "sroie": "dataset8/hf_dataset",
    "schema_dialog": "dataset9/default",
}


def shorten(value, limit=1200):
    if isinstance(value, dict):
        return {
            k: shorten(v, limit)
            for k, v in value.items()
        }

    if isinstance(value, list):
        return [shorten(v, limit) for v in value[:5]]

    if isinstance(value, str):
        return value[:limit]

    return value


def inspect_dataset(name, path):
    print(f"\n{'='*70}")
    print(f"DATASET: {name}")
    print(f"PATH: {path}")
    print("="*70)

    try:
        ds = load_from_disk(str(path))

        splits = (
            ds.items()
            if isinstance(ds, DatasetDict)
            else [("data", ds)]
        )

        result = {}

        for split_name, split in splits:
            print(f"\nSplit: {split_name}")
            print(f"Rows: {len(split)}")
            print(f"Columns: {split.column_names}")
            print(f"Features: {split.features}")

            samples = [
                shorten(split[i])
                for i in range(min(3, len(split)))
            ]

            result[split_name] = {
                "rows": len(split),
                "columns": split.column_names,
                "features": str(split.features),
                "samples": samples
            }

            print("Sample records:")
            for sample in samples:
                print(json.dumps(
                    sample,
                    indent=2,
                    ensure_ascii=False,
                    default=str
                )[:5000])

        return result

    except Exception as e:
        print(f"FAILED: {e}")
        return {"error": str(e)}


def inspect_jsonl_files():
    results = {}

    for path in Path("dataset5").glob("*.jsonl"):
        print(f"\nInspecting {path}")

        count = 0
        errors = 0
        samples = []

        with open(path, encoding="utf-8") as f:
            for line in f:
                if not line.strip():
                    continue

                try:
                    row = json.loads(line)
                    count += 1

                    if len(samples) < 3:
                        samples.append(shorten(row))

                except Exception:
                    errors += 1

        results[path.name] = {
            "rows": count,
            "errors": errors,
            "samples": samples
        }

        print(f"Rows: {count} | Errors: {errors}")


    return results


def main():
    report = {}

    for name, path in DATASETS.items():
        report[name] = inspect_dataset(name, path)

    report["synthetic_jsonl"] = inspect_jsonl_files()

    with open(
        "dataset_inspection.json",
        "w",
        encoding="utf-8"
    ) as f:
        json.dump(
            report,
            f,
            indent=2,
            ensure_ascii=False,
            default=str
        )

    print("\nInspection complete!")
    print("Saved: dataset_inspection.json")


if __name__ == "__main__":
    main()



DATASET: nemotron_direct
PATH: dataset2/direct_generation

Split: train
Rows: 28145
Columns: ['responses_create_params', 'schema_str', 'schema_type', 'schema_fields_count', 'agent_ref']
Features: {'responses_create_params': {'input': List({'content': Value('string'), 'role': Value('string')})}, 'schema_str': Value('string'), 'schema_type': Value('string'), 'schema_fields_count': Value('string'), 'agent_ref': {'type': Value('string'), 'name': Value('string')}}
Sample records:
{
  "responses_create_params": {
    "input": [
      {
        "content": "Response Format (XML with root <output>): {\"type\": \"object\", \"properties\": {\"output\": {\"type\": \"object\", \"required\": [\"petDetails\", \"ownerInformation\", \"reportMetadata\"], \"properties\": {\"petDetails\": {\"type\": \"object\", \"required\": [\"species\", \"breed\", \"identifyingFeatures\"], \"additionalProperties\": false, \"properties\": {\"species\": {\"type\": \"string\", \"enum\": [\"Dog\", \"Cat\", \"Bird\", \"Rabb

In [ ]:
from pathlib import Path
from collections import Counter
import json
import hashlib
import re

from datasets import load_from_disk, DatasetDict

ROOT = Path(".")
OUTPUT_JSON = ROOT / "quality_audit_report.json"
OUTPUT_TXT = ROOT / "quality_audit_summary.txt"

# Optional but recommended:
# pip install jsonschema
try:
    from jsonschema import Draft202012Validator
    HAS_JSONSCHEMA = True
except ImportError:
    HAS_JSONSCHEMA = False


# ---------------------------------------------------------
# DATASET CONFIGURATION
# ---------------------------------------------------------

DATASETS = {
    "nemotron_direct": ("dataset2/direct_generation", "sft_candidate"),
    "nemotron_diversified": ("dataset2/diversified_tasks", "sft_candidate"),
    "nemotron_tool_extraction": ("dataset2/tool_calling_extraction", "tool_candidate"),
    "emulated_pool": ("dataset3/hf_dataset", "mixed_candidate"),
    "structured_sft": ("dataset4/hf_dataset", "sft_candidate"),
    "sob": ("dataset6/hf_dataset", "sft_candidate"),
    "conll": ("dataset7/hf_dataset", "auxiliary"),
    "sroie": ("dataset8/hf_dataset", "auxiliary"),
    "schema_dialog": ("dataset9/default", "auxiliary"),
}

JSONL_FILES = {
    "dpo_function_calling": ("dataset5/dpo_function_calling.jsonl", "dpo"),
    "dpo_structured_output": ("dataset5/dpo_structured_output.jsonl", "dpo"),
    "sft_synthetic_extraction": ("dataset5/sft_synthetic_extraction.jsonl", "sft_candidate"),
    "sft_synthetic_json": ("dataset5/sft_synthetic_json.jsonl", "sft_candidate"),
}


# ---------------------------------------------------------
# HELPERS
# ---------------------------------------------------------

def as_text(value):
    if value is None:
        return None

    if isinstance(value, str):
        return value.strip() or None

    if isinstance(value, (dict, list)):
        return json.dumps(value, ensure_ascii=False)

    return str(value)


def get_first(row, keys):
    for key in keys:
        value = row.get(key)
        if value is not None and value != "":
            return value
    return None


def extract_messages(row):
    """Extract prompt and assistant response from common conversation formats."""

    messages = None

    params = row.get("responses_create_params")
    if isinstance(params, dict):
        messages = params.get("input")

    if messages is None:
        messages = (
            row.get("messages")
            or row.get("conversations")
            or row.get("conversation")
        )

    if not isinstance(messages, list):
        return None, None

    user_parts = []
    assistant_parts = []

    for message in messages:
        if not isinstance(message, dict):
            continue

        role = str(
            message.get("role")
            or message.get("from")
            or message.get("speaker")
            or ""
        ).lower()

        content = (
            message.get("content")
            or message.get("value")
            or message.get("text")
        )

        content = as_text(content)
        if not content:
            continue

        if role in ("user", "human"):
            user_parts.append(content)
        elif role in ("assistant", "gpt", "bot"):
            assistant_parts.append(content)

    prompt = "\n\n".join(user_parts) if user_parts else None
    target = "\n\n".join(assistant_parts) if assistant_parts else None

    return prompt, target


def extract_fields(row):
    """Best-effort extraction without assuming every dataset has the same format."""

    prompt, target = extract_messages(row)

    if prompt is None:
        prompt = as_text(get_first(row, [
            "prompt",
            "request",
            "question",
            "instruction",
            "input",
        ]))

    if target is None:
        target = as_text(get_first(row, [
            "validated_output",
            "ground_truth",
            "output",
            "response",
            "completion",
            "target",
            "answer",
            "chosen",
        ]))

    schema = as_text(get_first(row, [
        "schema",
        "json_schema",
        "schema_str",
        "output_schema",
    ]))

    # Some records store the source document separately.
    document = as_text(get_first(row, [
        "document",
        "context",
        "source_text",
    ]))

    if prompt is None and document:
        prompt = document

    return prompt, target, schema


def clean_json_string(value):
    """Remove common Markdown code fences before attempting JSON parsing."""

    if not isinstance(value, str):
        return value

    value = value.strip()

    match = re.fullmatch(
        r"```(?:json)?\s*(.*?)\s*```",
        value,
        flags=re.DOTALL | re.IGNORECASE
    )

    if match:
        value = match.group(1).strip()

    return value


def parse_json(value):
    if value is None:
        return False, None

    try:
        cleaned = clean_json_string(value)

        if isinstance(cleaned, (dict, list)):
            return True, cleaned

        return True, json.loads(cleaned)

    except (ValueError, TypeError):
        return False, None


def validate_against_schema(target, schema):
    if not HAS_JSONSCHEMA or target is None or schema is None:
        return None

    schema_ok, schema_obj = parse_json(schema)
    target_ok, target_obj = parse_json(target)

    if not schema_ok or not target_ok:
        return None

    if not isinstance(schema_obj, dict):
        return None

    try:
        Draft202012Validator.check_schema(schema_obj)
        validator = Draft202012Validator(schema_obj)

        return not any(validator.iter_errors(target_obj))

    except Exception:
        return None


def fingerprint(prompt, target, schema):
    content = json.dumps(
        [prompt, target, schema],
        ensure_ascii=False,
        sort_keys=True,
        default=str
    )
    return hashlib.sha256(content.encode("utf-8")).hexdigest()


def inspect_row(row, index, seen_hashes):
    prompt, target, schema = extract_fields(row)

    prompt_present = bool(prompt)
    target_present = bool(target)
    schema_present = bool(schema)

    target_json, _ = parse_json(target)
    schema_json, _ = parse_json(schema)

    schema_valid_target = validate_against_schema(target, schema)

    # Duplicate detection is only meaningful when we have useful content.
    duplicate = False
    if prompt_present or target_present or schema_present:
        key = fingerprint(prompt, target, schema)
        duplicate = key in seen_hashes
        seen_hashes.add(key)

    issues = []

    if not prompt_present:
        issues.append("missing_prompt")

    if not target_present:
        issues.append("missing_target")

    if not schema_present:
        issues.append("missing_schema")

    if target_present and not target_json:
        issues.append("target_not_json")

    if schema_present and not schema_json:
        issues.append("schema_not_json")

    if schema_valid_target is False:
        issues.append("schema_validation_failed")

    if duplicate:
        issues.append("duplicate")

    return {
        "prompt_present": prompt_present,
        "target_present": target_present,
        "schema_present": schema_present,
        "target_json_parseable": target_json if target_present else None,
        "schema_json_parseable": schema_json if schema_present else None,
        "schema_valid_target": schema_valid_target,
        "duplicate": duplicate,
        "issues": issues,
    }


# ---------------------------------------------------------
# REPORTING
# ---------------------------------------------------------

def new_stats():
    return {
        "rows": 0,
        "prompt_present": 0,
        "target_present": 0,
        "schema_present": 0,
        "target_json_parseable": 0,
        "schema_json_parseable": 0,
        "schema_valid_target": 0,
        "schema_validation_failed": 0,
        "duplicates": 0,
        "issue_counts": Counter(),
        "examples_with_issues": [],
    }


def update_stats(stats, result, index):
    stats["rows"] += 1

    for field in (
        "prompt_present",
        "target_present",
        "schema_present",
        "target_json_parseable",
        "schema_json_parseable",
        "duplicate",
    ):
        if result[field]:
            stats[field] += 1

    if result["schema_valid_target"] is True:
        stats["schema_valid_target"] += 1

    if result["schema_valid_target"] is False:
        stats["schema_validation_failed"] += 1

    for issue in result["issues"]:
        stats["issue_counts"][issue] += 1

    if result["issues"] and len(stats["examples_with_issues"]) < 20:
        stats["examples_with_issues"].append({
            "row_index": index,
            "issues": result["issues"],
        })


def finalize_stats(stats):
    result = dict(stats)
    result["issue_counts"] = dict(stats["issue_counts"])
    return result


def inspect_split(split):
    stats = new_stats()
    seen_hashes = set()

    for index, row in enumerate(split):
        result = inspect_row(row, index, seen_hashes)
        update_stats(stats, result, index)

    return finalize_stats(stats)


# ---------------------------------------------------------
# DATASET LOADERS
# ---------------------------------------------------------

def inspect_saved_dataset(name, relative_path, category):
    path = ROOT / relative_path

    if not path.exists():
        return {
            "category": category,
            "error": f"Path not found: {path}",
        }

    try:
        dataset = load_from_disk(str(path))

        if isinstance(dataset, DatasetDict):
            splits = {
                split_name: inspect_split(split)
                for split_name, split in dataset.items()
            }
        else:
            splits = {
                "data": inspect_split(dataset)
            }

        return {
            "category": category,
            "splits": splits,
        }

    except Exception as exc:
        return {
            "category": category,
            "error": str(exc),
        }


def inspect_jsonl(name, relative_path, category):
    path = ROOT / relative_path

    if not path.exists():
        return {
            "category": category,
            "error": f"Path not found: {path}",
        }

    stats = new_stats()
    seen_hashes = set()
    malformed_lines = []

    with open(path, "r", encoding="utf-8") as file:
        for index, line in enumerate(file):
            if not line.strip():
                continue

            try:
                row = json.loads(line)

                if not isinstance(row, dict):
                    raise ValueError("Record is not a JSON object")

                result = inspect_row(row, index, seen_hashes)
                update_stats(stats, result, index)

            except Exception as exc:
                malformed_lines.append({
                    "line": index + 1,
                    "error": str(exc),
                })

    return {
        "category": category,
        "splits": {
            "data": finalize_stats(stats)
        },
        "malformed_lines": malformed_lines[:50],
        "malformed_line_count": len(malformed_lines),
    }


# ---------------------------------------------------------
# MAIN
# ---------------------------------------------------------

def main():
    print("=" * 65)
    print("STRUCTURED OUTPUT DATASET QUALITY AUDIT")
    print("=" * 65)

    report = {
        "jsonschema_available": HAS_JSONSCHEMA,
        "datasets": {},
    }

    for name, (path, category) in DATASETS.items():
        print(f"\nInspecting {name} ...")
        report["datasets"][name] = inspect_saved_dataset(
            name, path, category
        )

    for name, (path, category) in JSONL_FILES.items():
        print(f"\nInspecting {name} ...")
        report["datasets"][name] = inspect_jsonl(
            name, path, category
        )

    with open(OUTPUT_JSON, "w", encoding="utf-8") as file:
        json.dump(report, file, indent=2, ensure_ascii=False)

    lines = [
        "STRUCTURED OUTPUT DATASET QUALITY AUDIT",
        "=" * 65,
        f"JSON Schema validation available: {HAS_JSONSCHEMA}",
        "",
    ]

    for name, dataset_info in report["datasets"].items():
        lines.append(f"\nDATASET: {name}")
        lines.append(f"Category: {dataset_info.get('category')}")

        if "error" in dataset_info:
            lines.append(f"ERROR: {dataset_info['error']}")
            continue

        for split_name, stats in dataset_info.get("splits", {}).items():
            rows = stats["rows"]

            lines.extend([
                f"\n  Split: {split_name}",
                f"  Rows: {rows}",
            ])

            for field in (
                "prompt_present",
                "target_present",
                "schema_present",
                "target_json_parseable",
                "schema_json_parseable",
                "schema_valid_target",
                "schema_validation_failed",
                "duplicates",
            ):
                value = stats[field]
                percentage = (value / rows * 100) if rows else 0

                lines.append(
                    f"  {field}: {value} ({percentage:.2f}%)"
                )

            lines.append("  Issues:")
            for issue, count in stats["issue_counts"].items():
                lines.append(f"    {issue}: {count}")

        if dataset_info.get("malformed_line_count", 0):
            lines.append(
                f"\n  Malformed JSONL lines: "
                f"{dataset_info['malformed_line_count']}"
            )

    with open(OUTPUT_TXT, "w", encoding="utf-8") as file:
        file.write("\n".join(lines))

    print("\n" + "=" * 65)
    print("AUDIT COMPLETE")
    print("=" * 65)
    print(f"JSON report: {OUTPUT_JSON}")
    print(f"Text summary: {OUTPUT_TXT}")


if __name__ == "__main__":
    main()


STRUCTURED OUTPUT DATASET QUALITY AUDIT

Inspecting nemotron_direct ...
